# Pareja B · Idea 4 — Focal Loss

La BCE actual cuenta igual los miles de "no es Misterio" obvios que los casos confusos. Focal Loss multiplica cada error por (1 − p)^γ: lo fácil casi no cuenta (p = 0.9 → ×0.01 con γ = 2) y el entrenamiento se va a los confusos (p = 0.3 → ×0.49). Probar γ = 1, 2, 3; los umbrales se recalibran solos al entrenar.

**Dónde se programa:** `perdidas.py`: el tipo `focal` (hoy lanza NotImplementedError). La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** su prueba pasa (quiten el `xfail` de `test_idea4` en
`tests/test_pendientes.py`) y el resultado en validación está en el registro.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).

## Preparar el entorno

**CPU basta**: entrenar una cabeza con vectores ya calculados tarda segundos.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`B-focal`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "B-focal"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea4 -rxX

## Experimentos (`experimentos/B4_focal_g1.yaml`, `_g2`, `_g3`)

In [ ]:
# entrenar en train, evaluar en VALIDACIÓN y comparar contra base (los 4 débiles salen primero)
for nombre in ["B4_focal_g1", "B4_focal_g2", "B4_focal_g3"]:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
    !python -m clasificador.comparar base {nombre}

## Subir a la rama `B-focal`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`B-focal`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Fine-grained tokens*).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")